# Week 37, Day 5: Nested data: neurons, sessions and animals

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w037/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: False positives by the hundred

Write experiment(), returning the two p-values (naive across neurons, and across mouse means) for one simulated experiment with no true group difference. Mouse means vary with standard deviation 0.3, sessions within a mouse with 0.2, neurons within a session with 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
r = np.random.default_rng(2014)
n_mice, n_sess, n_neur = 5, 4, 30             # per group: 5 mice x 4 sessions x 30 neurons = 600 neurons

def simulate_group():
    vals, mouse = [], []
    for m in range(n_mice):
        mu_m = r.normal(0, 0.3)
        for s in range(n_sess):
            mu_s = mu_m + r.normal(0, 0.2)
            vals += list(mu_s + r.normal(0, 1, n_neur)); mouse += [m] * n_neur
    return np.array(vals), np.array(mouse)

def experiment():
    a, ma = simulate_group(); b, mb = simulate_group()
    p_neurons = ...
    p_mice = ...
    return p_neurons, p_mice

ps = np.array([experiment() for _ in range(300)])
fpr = {"neurons": np.mean(ps[:, 0] < 0.05), "mice": np.mean(ps[:, 1] < 0.05)}
print(f"false positive rate: across neurons {fpr['neurons']:.0%}, across mouse means {fpr['mice']:.0%}")

In [ ]:
import numpy as np
def test_naive():
    "a t-test across neurons gives false positives in more than 30% of experiments"
    assert fpr["neurons"] > 0.3

def test_mouse():
    "a t-test on mouse means keeps the false positive rate below 10%"
    assert fpr["mice"] < 0.1

test_naive()
test_mouse()
print("All checks passed")

### Exercise: Intraclass correlation and effective sample size

Estimate the ICC from many mice by the method of moments: the variance of the mouse means minus the within-mouse variance divided by m gives the between-mouse variance. Then compute the effective sample size of 600 neurons from 5 mice of 120 neurons each.

In [ ]:
var_between = 0.3 ** 2 + 0.2 ** 2 / n_sess                   # mouse plus averaged session variance
var_within = 1.0 + 0.2 ** 2 * (1 - 1 / n_sess)                # approximately, ignoring small terms
icc_true = var_between / (var_between + var_within)
vals, mouse = [], []
for _ in range(40):                                           # 40 groups of 5 mice, to estimate well
    v, m = simulate_group(); vals.append(v); mouse.append(m + 5 * len(mouse))
vals, mouse = np.concatenate(vals), np.concatenate(mouse)
m_per = n_sess * n_neur
means = np.array([vals[mouse == k].mean() for k in np.unique(mouse)])
within = np.mean([vals[mouse == k].var(ddof=1) for k in np.unique(mouse)])
sb2 = ...
icc_hat = sb2 / (sb2 + within)
n_eff = ...
print(f"ICC: true {icc_true:.3f}, estimated {icc_hat:.3f}; 600 neurons from 5 mice are worth {n_eff:.1f} independent neurons")

In [ ]:
import numpy as np
def test_icc():
    "the estimated ICC is within 0.05 of the true value"
    assert abs(icc_hat - icc_true) < 0.05

def test_neff():
    "600 neurons from 5 mice are worth fewer than 100 independent neurons"
    assert n_eff < 100

test_icc()
test_neff()
print("All checks passed")

### Exercise: An exact permutation test on animals

Write perm_test(means_a, means_b): pool the ten animal means, compute the difference of group means for every way of choosing five animals as group A (itertools.combinations), and return the observed difference, the list of null differences, and the two-sided p-value (the fraction of null differences at least as large in absolute value as the observed one).

In [ ]:
from itertools import combinations

def perm_test(means_a, means_b):
    pooled = np.r_[means_a, means_b]; n = len(means_a)
    obs = np.mean(means_a) - np.mean(means_b)
    null = ...
    p = ...
    return obs, null, p

def group_means(shift):
    v, m = simulate_group()
    return np.array([v[m == k].mean() for k in range(n_mice)]) + shift

obs, null, p_effect = perm_test(group_means(1.0), group_means(0.0))
_, _, p_null = perm_test(group_means(0.0), group_means(0.0))
print(f"with a true difference of 1.0: observed {obs:.2f}, p = {p_effect:.3f};  with none: p = {p_null:.3f}")
plt.hist(null, 30); plt.axvline(obs, color="k"); plt.xlabel("difference of group means under relabelling"); plt.show()

In [ ]:
import numpy as np
def test_exact():
    "the null distribution has one value for each of the 252 splits"
    assert len(null) == 252

def test_p():
    "the real effect is detected (p below 0.05) and the null data are not"
    assert p_effect < 0.05 and p_null > 0.05

test_exact()
test_p()
print("All checks passed")